Mistral Large 3 is Mistral’s first mixture-of-experts model since the seminal Mixtral series, and represents a substantial step forward in pretraining at Mistral. After post-training, the model achieves parity with the best instruction-tuned open-weight models on the market on general prompts, while also demonstrating image understanding and best-in-class performance on multilingual conversations (i.e., non-English/Chinese).

Mistral Large 3 debuts at #2 in the OSS non-reasoning models category (#6 amongst OSS models overall) on the <a href="https://lmarena.ai/leaderboard/text">LMArena leaderboard</a> and is released under the Apache 2.0 license.

In this notebook, we will show how some basics on how to work with Mistral Large 3 in Microsoft Foundry, with a focus on multi-modal and financial data, to ask questions about the pdfs.

In [2]:
import base64
import json
import requests
import os
from dotenv import load_dotenv
from typing import Dict, Any
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool
from openai.types.responses.response_input_param import McpApprovalResponse, ResponseInputParam


from IPython.display import Image, display, IFrame

In [3]:

load_dotenv()

AZURE_MISTRAL_MODEL_ENDPOINT = os.getenv("AZURE_MISTRAL_MODEL_ENDPOINT")
AZURE_MISTRAL_MODEL_KEY = os.getenv("AZURE_MISTRAL_MODEL_KEY")

AZURE_AI_PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")

AZURE_AI_DEPLOYMENT_NAME = os.getenv("AZURE_AI_DEPLOYMENT_NAME")

REQUEST_HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {AZURE_MISTRAL_MODEL_KEY}",
}

In [4]:
def encode_file(file_path: str) -> str:
    try:
        with open(file_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode("utf-8")
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
        return None


In [5]:
pdf_path = "cvna_2021_annual_report.png"
cvna_2021_annual_report = encode_file(pdf_path)

In [6]:
def documentPayload(encoded_document: str, question: str) -> dict[str, Any]:
  ## This is a sample payload for sending a document to the Mistral Document AI API. The "content" field contains a list of message parts, which can include text and images. In this example, we are sending a text prompt along with an image encoded in base64 format. The model specified in the payload will process the input and generate a response based on the content of the document.
  
  payload = {
    "model": f"{AZURE_AI_DEPLOYMENT_NAME}",
    "messages": [
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": question
          },
         {
              "type": "image_url",
                "image_url": {
                      "url": f"data:image/png;base64,{encoded_document}"
                }
         }
        ]
      }
    ]
  }
  return payload

def textPayload(question: str) -> dict[str, Any]:
  ## This is a sample payload for sending a text prompt to the Mistral Document AI API. The "content" field contains a list of message parts, which in this case includes only a text prompt. The model specified in the payload will process the input and generate a response based on the provided text.
  
  payload = {
    "model": f"{AZURE_AI_DEPLOYMENT_NAME}",
    "messages": [
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": question
          }
        ]
      }
    ]
  }
  return payload


In [7]:
def send_document_ai_request(encoded_document: str, question: str) -> dict:
    if not encoded_document:
        payload = textPayload(question)
    else:
        payload = documentPayload(encoded_document, question)
    response = requests.post(
        url=AZURE_MISTRAL_MODEL_ENDPOINT, 
        json=payload,
        headers=REQUEST_HEADERS,
    )
    return response.json()
documentResponse = send_document_ai_request(cvna_2021_annual_report, "What's in this document? Answer in a single sentence?")

In [8]:
display(documentResponse['choices'][0]['message']['content'])

'The image presents data from the Carvana 2021 Annual Report, showcasing four key metrics over the years 2014 to 2021: Retail Units Sold, Total Revenue, Total Markets at Year End, and Car Vending Machines. Specifically, it highlights a significant growth trend in each category, with retail units sold increasing to 425,237, total revenue rising to $12.814 billion, total markets expanding to 311, and car vending machines reaching 30 by the end of 2021.'

In [9]:
questions = [
    "What was CVNA revenue in 2020?",
    "How many additional markets has Carvana added since 2014?",
    "What was 2016 revenue per retail unit sold?",
]

for index, question in enumerate(questions):
    documentResponse = send_document_ai_request(cvna_2021_annual_report, question)
    display(documentResponse['choices'][0]['message']['content'])

'The chart for Total Revenue shows the value for 2020 as $3,940 million.\n\nAs a result, the final answer is: $3,940 million.'

'Carvana had 3 markets at the end of 2014 and 311 markets at the end of 2021. The additional markets added are calculated as 311 - 3.\n\nAs a result, the final answer is: 308.'

'The total revenue for 2016 is $130 million, and the retail units sold in 2016 are 18,761. To find the revenue per retail unit, divide the total revenue by the number of units sold: $130,000,000 / 18,761 ≈ $6,929.\n\nAs a result, the answer is: $6,929.'